# Task 1 Extension — Zero-Shot Sentiment Classification with Nemotron 3 Ultra

This experiment evaluates NVIDIA Nemotron 3 Ultra as a zero-shot sentiment classifier on Amazon customer reviews.

The model is accessed through NVIDIA's hosted API and is not fine-tuned on the project dataset.

### Sentiment Labels
- **Negative:** 1–2 stars
- **Neutral:** 3 stars
- **Positive:** 4–5 stars

### Goal
Compare the zero-shot performance of Nemotron 3 Ultra with the previously trained traditional machine-learning and fine-tuned Transformer models, with particular attention to Macro F1 and minority-class performance.

In [8]:
!pip install -q openai

In [11]:
from google.colab import userdata
from openai import OpenAI

NVIDIA_API_KEY = userdata.get("NVIDIA_API_KEY")

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key=NVIDIA_API_KEY
)

print("NVIDIA client ready.")

NVIDIA client ready.


In [12]:
response = client.chat.completions.create(
    model="nvidia/nemotron-3-ultra-550b-a55b",
    messages=[
        {
            "role": "system",
            "content": """
You are a sentiment classifier.

Classify the customer review into exactly one label:
Negative
Neutral
Positive

Return ONLY the label.
"""
        },
        {
            "role": "user",
            "content": """
I love this tablet. The screen is great and the battery lasts a long time.
"""
        }
    ],
    temperature=0,
    max_tokens=10,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    }
)

prediction = response.choices[0].message.content.strip()

print("Prediction:", prediction)

Prediction: Positive


In [19]:
import glob
from google.colab import drive

drive.mount('/content/drive')
matches = glob.glob(
    "/content/drive/MyDrive/**/*cleaned*reviews*.csv",
    recursive=True
)

print("Files found:")
for file in matches:
    print(file)

Mounted at /content/drive
Files found:
/content/drive/MyDrive/MahmoodaProjectNLP/data/cleaned_reviews_mahmooda.csv


In [20]:
import pandas as pd

DATA_PATH = "/content/drive/MyDrive/MahmoodaProjectNLP/data/cleaned_reviews_mahmooda.csv"

df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())


Dataset shape: (34626, 15)

Columns:
['id', 'name', 'asins', 'brand', 'categories', 'manufacturer', 'reviews.date', 'reviews.doRecommend', 'reviews.numHelpful', 'reviews.rating', 'reviews.text', 'reviews.title', 'text', 'review_word_count', 'sentiment']


In [21]:
# Check available labels first
print(df["sentiment"].value_counts())

# Create a balanced evaluation sample
nemotron_sample = (
    df.groupby("sentiment", group_keys=False)
      .sample(n=100, random_state=42)
      .reset_index(drop=True)
)

# Shuffle the rows
nemotron_sample = nemotron_sample.sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

print("\nNemotron evaluation sample:")
print(nemotron_sample["sentiment"].value_counts())

print("\nTotal reviews:", len(nemotron_sample))

sentiment
Positive    32315
Neutral      1499
Negative      812
Name: count, dtype: int64

Nemotron evaluation sample:
sentiment
Positive    100
Neutral     100
Negative    100
Name: count, dtype: int64

Total reviews: 300


In [24]:
def classify_with_nemotron(review_text):
    response = client.chat.completions.create(
        model="nvidia/nemotron-3-ultra-550b-a55b",
        messages=[
            {
                "role": "system",
                "content": """
You are a sentiment classifier for Amazon customer reviews.

Classify each review into exactly ONE of these labels:

Negative
Neutral
Positive

Use these definitions:
Negative = clearly dissatisfied, critical, or unfavorable.
Neutral = mixed, balanced, uncertain, or neither clearly positive nor negative.
Positive = clearly satisfied, approving, or favorable.

Return ONLY the label.
Do not explain your reasoning.
"""
            },
            {
                "role": "user",
                "content": str(review_text)
            }
        ],
        temperature=0,
        max_tokens=10,
        extra_body={
            "chat_template_kwargs": {
                "enable_thinking": False
            }
        }
    )

    return response.choices[0].message.content.strip()

In [23]:
for label in ["Negative", "Neutral", "Positive"]:
    row = nemotron_sample[
        nemotron_sample["sentiment"] == label
    ].iloc[0]

    review = row["text"]
    prediction = classify_with_nemotron(review)

    print("=" * 70)
    print("Actual:    ", label)
    print("Prediction:", prediction)
    print("Review:    ", review[:300])
    print()

Actual:     Negative
Prediction: Negative
Review:     Never booted. I bought this thinking since we'd bought several other refurbished things that worked, that it would be okay. The box never booted properly so the only thing we could stare at was the Amazon logo and the words Unknown Boot in the corner. Customer Service tried to help with the box but 

Actual:     Neutral
Prediction: Positive
Review:     ok tablet for a beginner. good tablet for a young person to begin with small to handle

Actual:     Positive
Prediction: Positive
Review:     Great Tablet for the money. Used Kindle tablets before needed update. As sson as I started to use it liked it even more



In [26]:
results = pd.read_csv(SAVE_PATH)

print("Total rows:", len(results))

print("\nPrediction values:")
print(results["nemotron_prediction"].value_counts(dropna=False))

error_mask = results["nemotron_prediction"] == "ERROR"

print("\nTotal failed API calls:", error_mask.sum())
print("Failed row indices:")
print(results.index[error_mask].tolist())

Total rows: 300

Prediction values:
nemotron_prediction
ERROR       186
Positive     46
Negative     44
Neutral      24
Name: count, dtype: int64

Total failed API calls: 186
Failed row indices:
[8, 12, 21, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 50, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 74, 89, 90, 91, 92, 93, 94, 95, 96, 97, 98, 99, 107, 124, 152, 154, 155, 158, 159, 160, 161, 162, 164, 165, 166, 167, 168, 169, 170, 171, 172, 173, 174, 175, 176, 177, 178, 179, 180, 181, 182, 183, 184, 185, 186, 187, 188, 189, 190, 191, 192, 193, 194, 195, 196, 197, 198, 199, 200, 201, 202, 203, 204, 205, 206, 207, 208, 209, 210, 211, 212, 213, 214, 215, 216, 217, 218, 219, 220, 221, 222, 223, 224, 225, 226, 227, 228, 229, 230, 231, 232, 233, 234, 235, 236, 237, 238, 239, 240, 241, 242, 243, 244, 245, 246, 247, 248, 249, 250, 251, 252, 253, 254, 255, 256, 257, 258, 259, 260, 261, 262, 263, 264, 265, 266, 267, 268, 269, 270, 271, 272, 273, 274, 275, 276, 277, 278, 279, 280, 281, 282, 

In [27]:
import time
import pandas as pd

results = pd.read_csv(SAVE_PATH)

error_indices = results.index[
    results["nemotron_prediction"] == "ERROR"
].tolist()

print(f"Predictions to retry: {len(error_indices)}")

for count, i in enumerate(error_indices, start=1):

    success = False

    # Maximum 5 attempts for each review
    for attempt in range(5):

        try:
            prediction = classify_with_nemotron(
                results.loc[i, "text"]
            )

            if prediction in ["Negative", "Neutral", "Positive"]:
                results.loc[i, "nemotron_prediction"] = prediction
                success = True

                print(
                    f"[{count}/{len(error_indices)}] "
                    f"Row {i}: {prediction}"
                )

                break

        except Exception as e:

            # Increasing wait:
            # 15s → 30s → 60s → 120s → 240s
            wait_time = 15 * (2 ** attempt)

            print(
                f"Row {i} failed "
                f"(attempt {attempt + 1}/5). "
                f"Waiting {wait_time}s..."
            )

            time.sleep(wait_time)

    # Save immediately after each successful prediction
    if success:
        results.to_csv(SAVE_PATH, index=False)

    # Slow down normal requests too
    time.sleep(3)

print("\nRetry session finished.")

remaining = (
    results["nemotron_prediction"] == "ERROR"
).sum()

print("Remaining errors:", remaining)

results.to_csv(SAVE_PATH, index=False)

Predictions to retry: 186
Row 8 failed (attempt 1/5). Waiting 15s...
Row 8 failed (attempt 2/5). Waiting 30s...
[1/186] Row 8: Negative
[2/186] Row 12: Positive
[3/186] Row 21: Negative
Row 27 failed (attempt 1/5). Waiting 15s...
[4/186] Row 27: Positive
Row 28 failed (attempt 1/5). Waiting 15s...
[5/186] Row 28: Neutral
Row 29 failed (attempt 1/5). Waiting 15s...
Row 29 failed (attempt 2/5). Waiting 30s...
[6/186] Row 29: Positive
[7/186] Row 30: Negative
[8/186] Row 31: Neutral
[9/186] Row 32: Negative
[10/186] Row 33: Negative
Row 34 failed (attempt 1/5). Waiting 15s...
Row 34 failed (attempt 2/5). Waiting 30s...
Row 34 failed (attempt 3/5). Waiting 60s...
[11/186] Row 34: Positive
[12/186] Row 35: Neutral
[13/186] Row 36: Negative
Row 37 failed (attempt 1/5). Waiting 15s...
Row 37 failed (attempt 2/5). Waiting 30s...
[14/186] Row 37: Negative
Row 50 failed (attempt 1/5). Waiting 15s...
[15/186] Row 50: Neutral
[16/186] Row 58: Negative
[17/186] Row 59: Negative
[18/186] Row 60: Pos

In [28]:
import time
import pandas as pd

results = pd.read_csv(SAVE_PATH)

error_indices = results.index[
    results["nemotron_prediction"] == "ERROR"
].tolist()

print("Remaining error rows:", error_indices)

if len(error_indices) > 0:
    i = error_indices[0]

    print(f"Waiting before retrying row {i}...")
    time.sleep(30)

    try:
        prediction = classify_with_nemotron(
            results.loc[i, "text"]
        )

        if prediction in ["Negative", "Neutral", "Positive"]:
            results.loc[i, "nemotron_prediction"] = prediction
            results.to_csv(SAVE_PATH, index=False)
            print("Prediction:", prediction)

    except Exception as e:
        print("Retry failed:", e)

print(
    "Remaining errors:",
    (results["nemotron_prediction"] == "ERROR").sum()
)

Remaining error rows: [252]
Waiting before retrying row 252...
Prediction: Negative
Remaining errors: 0


In [29]:
from sklearn.metrics import (
    accuracy_score,
    f1_score,
    classification_report,
    confusion_matrix
)

# Ground truth and Nemotron predictions
y_true = results["sentiment"]
y_pred = results["nemotron_prediction"]

# Main metrics
accuracy = accuracy_score(y_true, y_pred)
macro_f1 = f1_score(y_true, y_pred, average="macro")

print("NEMOTRON 3 ULTRA — BALANCED 300-REVIEW EVALUATION")
print("=" * 60)

print(f"Accuracy : {accuracy:.4f}")
print(f"Macro F1 : {macro_f1:.4f}")

print("\nClassification Report:")
print(
    classification_report(
        y_true,
        y_pred,
        labels=["Negative", "Neutral", "Positive"],
        digits=4
    )
)

print("Confusion Matrix:")
cm = confusion_matrix(
    y_true,
    y_pred,
    labels=["Negative", "Neutral", "Positive"]
)

print(cm)

NEMOTRON 3 ULTRA — BALANCED 300-REVIEW EVALUATION
Accuracy : 0.7467
Macro F1 : 0.7244

Classification Report:
              precision    recall  f1-score   support

    Negative     0.7521    0.8800    0.8111       100
     Neutral     0.7593    0.4100    0.5325       100
    Positive     0.7364    0.9500    0.8297       100

    accuracy                         0.7467       300
   macro avg     0.7493    0.7467    0.7244       300
weighted avg     0.7493    0.7467    0.7244       300

Confusion Matrix:
[[88  8  4]
 [29 41 30]
 [ 0  5 95]]


In [30]:
test_reviews = [
    {
        "id": 1,
        "text": "The tablet keeps freezing and the battery dies quickly. I regret buying it."
    },
    {
        "id": 2,
        "text": "It's an okay tablet. Some things work well, but there are also a few problems."
    },
    {
        "id": 3,
        "text": "Excellent tablet. Easy to use, fast, and the battery lasts a long time."
    }
]

prompt = """
Classify each Amazon customer review into exactly one sentiment:

Negative
Neutral
Positive

Definitions:
Negative = clearly dissatisfied, critical, or unfavorable.
Neutral = mixed, balanced, lukewarm, or neither clearly positive nor negative.
Positive = clearly satisfied, approving, or favorable.

Return ONLY the three labels in this format:

1: Label
2: Label
3: Label

Reviews:

1: {review1}

2: {review2}

3: {review3}
""".format(
    review1=test_reviews[0]["text"],
    review2=test_reviews[1]["text"],
    review3=test_reviews[2]["text"]
)

response = client.chat.completions.create(
    model="nvidia/nemotron-3-ultra-550b-a55b",
    messages=[
        {"role": "user", "content": prompt}
    ],
    temperature=0,
    max_tokens=30,
    extra_body={
        "chat_template_kwargs": {
            "enable_thinking": False
        }
    }
)

print(response.choices[0].message.content)

1: Negative
2: Neutral
3: Positive


### Final Sentiment Analysis Conclusion

1) The experiments compared traditional machine-learning models, a fine-tuned
Transformer, and a large zero-shot language model.

2) Among the traditional approaches, Linear SVM achieved the strongest Macro F1
of 0.5625.

3) Fine-tuning DistilBERT substantially improved performance, reaching a Macro F1
of 0.6970 on the original held-out test set.

4) As an additional experiment, NVIDIA Nemotron 3 Ultra was evaluated as a
zero-shot sentiment classifier on a balanced sample of 300 reviews. It achieved
an accuracy of 0.7467 and a Macro F1 of 0.7244.

5) Nemotron performed particularly well on Negative reviews (F1 = 0.8111) and
Positive reviews (F1 = 0.8297), while Neutral remained the most difficult class
(F1 = 0.5325).

6) Because Nemotron was evaluated on a different, balanced diagnostic sample,
its Macro F1 of 0.7244 should not be interpreted as directly outperforming
DistilBERT. A same-sample evaluation is required for a controlled comparison.

7) DistilBERT therefore remains the primary fine-tuned sentiment model, while
Nemotron demonstrates the potential of large language models for zero-shot
sentiment classification.